# 03 Why a Bias Is There

| | |
|---|---|
| **Path** | Perceptron |
| **Prerequisite** | 02 |

Without a bias, the score is only $wx$. The boundary is forced through the origin. Two points on the same side of the origin can ask for opposite signs, and no single weight can serve both.

Take $x = 1$ with label pass, and $x = 2$ with label fail. Both measurements are positive.

A correct weight with no bias would need

$$
w\cdot 1 > 0 \qquad\text{and}\qquad w\cdot 2 < 0
$$

The first says $w > 0$. The second then says $2w > 0$, so $w\cdot 2$ cannot be negative. There is no such $w$.


## Example 1 — The updates do not discover a separator, because none exists

Start at $w = 0$ and update whenever the margin is not positive. The first few weights are

$$
0 \xrightarrow{\text{pass}} 1 \xrightarrow{\text{fail}} -1 \xrightarrow{\text{pass}} 0
$$

The weight is back at the start. The search can continue, and it will keep meeting a point on the wrong side.

### Predict

After six passes over the two points, a margin is still not positive, and the printed weights include a return to $0$.


In [1]:
# No bias. The two positive measurements want opposite signs, so a mistake remains.
weight = 0
points = [(1, 1), (2, -1)]
shown = []
for _ in range(6):
    for measurement, label in points:
        margin = label * (weight * measurement)
        if margin <= 0:
            weight = weight + label * measurement
            shown.append(weight)
margins = [label * (weight * measurement) for measurement, label in points]
print(shown)
print(margins)
assert 0 in shown
assert any(margin <= 0 for margin in margins)


[1, -1, 0, -2, -1, 0, -2, -1, 0, -2]
[-2, 4]


## Example 2 — A bias makes a candidate easy to check

Keep the same two points, and allow a bias. The candidate

$$
w = -2, \qquad b = 3
$$

scores them as

$$
\begin{aligned}
s(1) &= -2 + 3 = 1 > 0 \\
s(2) &= -4 + 3 = -1 < 0
\end{aligned}
$$

Pass at $1$, fail at $2$. Both margins equal $1$. Checking a candidate is two lines. It does not require a search.


In [2]:
# Hand: s(1) = 1 and s(2) = -1. Both margins are 1.
weight, bias = -2, 3
score_pass = weight * 1 + bias
score_fail = weight * 2 + bias
print(score_pass, score_fail)
assert score_pass == 1 and score_pass > 0
assert score_fail == -1 and score_fail < 0
assert (1) * score_pass == 1
assert (-1) * score_fail == 1


1 -1


## Example 3 — The search from zero is longer than the check

The same update rule, now with a bias, does reach $(w, b) = (-2, 3)$. The first four updates are

$$
\begin{aligned}
(0, 0) &\rightarrow (1, 1) \\
(1, 1) &\rightarrow (-1, 0) \\
(-1, 0) &\rightarrow (0, 1) \\
(0, 1) &\rightarrow (-2, 0)
\end{aligned}
$$

The walk is not finished there. Continuing the same rule takes $13$ updates in all.

### Predict

The cell prints `13` and the final pair `(-2, 3)`.


In [3]:
# The first four pairs are hand work. The loop finishes the search at (-2, 3).
weight, bias = 0, 0
points = [(1, 1), (2, -1)]
steps = []
for _ in range(30):
    changed = False
    for measurement, label in points:
        if label * (weight * measurement + bias) <= 0:
            weight = weight + label * measurement
            bias = bias + label
            steps.append((weight, bias))
            changed = True
    if not changed:
        break
print(steps[:4])
print(len(steps))
print(weight, bias)
assert steps[:4] == [(1, 1), (-1, 0), (0, 1), (-2, 0)]
assert len(steps) == 13
assert (weight, bias) == (-2, 3)


[(1, 1), (-1, 0), (0, 1), (-2, 0)]
13
-2 3


## Pitfall — A long walk is not a proof that a line exists

Example 1 walked and never arrived, and the short proof says why. Example 3 walked and arrived, and the two-line check says the destination is legal. Existence is a candidate that scores every point on the correct side. The walk is one way to look for a candidate.


## Summary

The bias is the permission to put the boundary somewhere other than the origin. Without it, two opposite labels on the positive axis are impossible. With it, $w = -2$ and $b = 3$ separate $x = 1$ from $x = 2$, and the search from zero reaches that pair in $13$ updates.
